In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

BASE2IDX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
    "N": 4  # optional
}

def encode_seq(seq, max_len=1000):
    seq = seq.strip().upper()
    ids = [BASE2IDX.get(b, 4) for b in seq]  # unknown -> 4
    if len(ids) > max_len:
        ids = ids[:max_len]
    elif len(ids) < max_len:
        ids = ids + [4] * (max_len - len(ids))  # pad with N
    return torch.tensor(ids, dtype=torch.long)

class PromoterDataset(Dataset):
    def __init__(self, fasta_path, max_len=1000):
        self.max_len = max_len
        self.seqs = []
        with open(fasta_path) as f:
            cur = []
            for line in f:
                if line.startswith(">"):
                    if cur:
                        self.seqs.append("".join(cur))
                        cur = []
                else:
                    cur.append(line.strip())
            if cur:
                self.seqs.append("".join(cur))

        self.encoded = [encode_seq(s, max_len=self.max_len) for s in self.seqs]

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, idx):
        x = self.encoded[idx]
        # input and target are the same for reconstruction
        return x, x


In [16]:
import torch.nn as nn
import torch.nn.functional as F

class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        """
        x: (B, L, D)
        returns: (B, L, D) with position embeddings added
        """
        B, L, D = x.size()
        positions = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        pos_emb = self.pe(positions)
        return x + pos_emb


In [17]:
class PromoterVAE(nn.Module):
    def __init__(
        self,
        vocab_size=5,      # A,C,G,T,N
        max_len=1000,
        d_model=256,
        n_heads=8,
        num_layers_enc=3,
        num_layers_dec=3,
        latent_dim=64,
        dropout=0.1
    ):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.latent_dim = latent_dim

        # ---- Encoder ----
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb_enc = PositionalEmbedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_logvar = nn.Linear(d_model, latent_dim)

        # ---- Decoder ----
        # We'll reconstruct full sequence from z by expanding it to L positions
        self.fc_z_to_seq = nn.Linear(latent_dim, d_model * max_len)

        self.pos_emb_dec = PositionalEmbedding(max_len, d_model)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers_dec)

        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, x):
        """
        x: (B, L) integer tokens
        """
        emb = self.token_emb(x)                # (B, L, D)
        emb = self.pos_emb_enc(emb)           # add positional
        h = self.encoder(emb)                 # (B, L, D)
        # pool over length -> global representation
        h_mean = h.mean(dim=1)                # (B, D)
        mu = self.fc_mu(h_mean)               # (B, latent_dim)
        logvar = self.fc_logvar(h_mean)       # (B, latent_dim)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        """
        z: (B, latent_dim)
        return: logits (B, L, vocab_size)
        """
        B = z.size(0)
        seq_h = self.fc_z_to_seq(z)           # (B, L*D)
        seq_h = seq_h.view(B, self.max_len, self.d_model)  # (B, L, D)
        seq_h = self.pos_emb_dec(seq_h)
        h_dec = self.decoder(seq_h)           # (B, L, D)
        logits = self.fc_out(h_dec)           # (B, L, V)
        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


In [18]:
class PromoterVAE(nn.Module):
    def __init__(
        self,
        vocab_size=5,      # A,C,G,T,N
        max_len=1000,
        d_model=256,
        n_heads=8,
        num_layers_enc=3,
        num_layers_dec=3,
        latent_dim=64,
        dropout=0.1
    ):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.latent_dim = latent_dim

        # ---- Encoder ----
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb_enc = PositionalEmbedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_logvar = nn.Linear(d_model, latent_dim)

        # ---- Decoder ----
        # We'll reconstruct full sequence from z by expanding it to L positions
        self.fc_z_to_seq = nn.Linear(latent_dim, d_model * max_len)

        self.pos_emb_dec = PositionalEmbedding(max_len, d_model)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers_dec)

        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, x):
        """
        x: (B, L) integer tokens
        """
        emb = self.token_emb(x)                # (B, L, D)
        emb = self.pos_emb_enc(emb)           # add positional
        h = self.encoder(emb)                 # (B, L, D)
        # pool over length -> global representation
        h_mean = h.mean(dim=1)                # (B, D)
        mu = self.fc_mu(h_mean)               # (B, latent_dim)
        logvar = self.fc_logvar(h_mean)       # (B, latent_dim)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        """
        z: (B, latent_dim)
        return: logits (B, L, vocab_size)
        """
        B = z.size(0)
        seq_h = self.fc_z_to_seq(z)           # (B, L*D)
        seq_h = seq_h.view(B, self.max_len, self.d_model)  # (B, L, D)
        seq_h = self.pos_emb_dec(seq_h)
        h_dec = self.decoder(seq_h)           # (B, L, D)
        logits = self.fc_out(h_dec)           # (B, L, V)
        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


In [28]:
def vae_loss(logits, targets, mu, logvar, kl_weight=1.0, free_bits=0.05):
    B, L, V = logits.size()

    # reconstruction loss
    logits_flat = logits.view(B*L, V)
    targets_flat = targets.view(B*L)
    recon_loss = F.cross_entropy(logits_flat, targets_flat)

    # KL per latent dimension
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())  # (B, latent_dim)
    kl_per_dim = kl_per_dim.mean(dim=0)                          # (latent_dim,)

    # apply free bits (force KL >= free_bits)
    kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)

    kl = kl_per_dim.sum()  # sum over latent dims

    loss = recon_loss + kl_weight * kl
    return loss, recon_loss, kl


In [29]:
from torch.optim import Adam

fasta_path = "top10000_promoters_raw_1000bp.fa"
max_len = 1000
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

from torch.utils.data import random_split

dataset = PromoterDataset(fasta_path, max_len=max_len)

val_ratio = 0.1
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print("Train size:", train_size)
print("Val size:", val_size)


model = PromoterVAE(
    vocab_size=5,
    max_len=max_len,
    d_model=256,
    n_heads=8,
    num_layers_enc=3,
    num_layers_dec=3,
    latent_dim=64,
    dropout=0.1
).to(device)

optimizer = Adam(model.parameters(), lr=1e-4)

def evaluate(model, loader, kl_weight=1.0):
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    N = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, mu, logvar = model(x)
            loss, recon_loss, kl = vae_loss(logits, y, mu, logvar, kl_weight)

            total_loss += loss.item() * x.size(0)
            total_recon += recon_loss.item() * x.size(0)
            total_kl += kl.item() * x.size(0)
            N += x.size(0)

    return (
        total_loss / N,
        total_recon / N,
        total_kl / N
    )

n_epochs = 10

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0

    kl_weight = min(1.0, epoch / 10.0)  # KL annealing

    # ---- Training ----
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits, mu, logvar = model(x)
        loss, recon_loss, kl = vae_loss(logits, y, mu, logvar, kl_weight)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_recon += recon_loss.item() * x.size(0)
        total_kl += kl.item() * x.size(0)

    train_loss = total_loss / train_size
    train_recon = total_recon / train_size
    train_kl = total_kl / train_size

    # ---- Validation ----
    val_loss, val_recon, val_kl = evaluate(model, val_loader, kl_weight)

    print(
        f"[Epoch {epoch+1}/{n_epochs}] "
        f"Train: loss={train_loss:.4f} recon={train_recon:.4f} kl={train_kl:.4f} | "
        f"Val: loss={val_loss:.4f} recon={val_recon:.4f} kl={val_kl:.4f} | "
        f"KLw={kl_weight:.2f}"
    )


Train size: 8994
Val size: 999
[Epoch 1/10] Train: loss=1.3790 recon=1.3790 kl=100.0298 | Val: loss=1.3602 recon=1.3602 kl=139.2797 | KLw=0.00
[Epoch 2/10] Train: loss=1.9190 recon=1.3834 kl=5.3564 | Val: loss=1.6981 recon=1.3780 kl=3.2011 | KLw=0.10


KeyboardInterrupt: 